## Task 1: Data Exploration and Preprocessing

1.Load the blogs_categories.csv dataset.

2.Inspect the dataset structure (shape, missing values, class distribution).

3.Clean the text in the Data column by converting to lowercase, removing punctuation and special characters, tokenizing, and removing stop words.

4.Convert the cleaned text into numerical features using TfidfVectorizer.


In [2]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

In [3]:
# 1. Load dataset
df = pd.read_csv("blogs.csv")

# 2. Data Exploration
print("Dataset Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nClass Distribution:\n", df["Labels"].value_counts())

# 3. Data Cleaning & Preprocessing
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters, digits, and punctuation
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize and remove stop words / short words
    tokens = text.split()
    tokens = [w for w in tokens if w not in ENGLISH_STOP_WORDS and len(w) > 2]
    return " ".join(tokens)  # Added space between quotes

# Fixed column name 'Data' (capitalized) and function name spelling
df['Clean_Data'] = df['Data'].apply(preprocess_text)

# 4. Feature Extraction (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['Clean_Data'])

print("\nTF-IDF Matrix Shape:", X_tfidf.shape)

Dataset Shape: (2000, 2)

Missing values:
 Data      0
Labels    0
dtype: int64

Class Distribution:
 Labels
alt.atheism                 100
comp.graphics               100
comp.os.ms-windows.misc     100
comp.sys.ibm.pc.hardware    100
comp.sys.mac.hardware       100
comp.windows.x              100
misc.forsale                100
rec.autos                   100
rec.motorcycles             100
rec.sport.baseball          100
rec.sport.hockey            100
sci.crypt                   100
sci.electronics             100
sci.med                     100
sci.space                   100
soc.religion.christian      100
talk.politics.guns          100
talk.politics.mideast       100
talk.politics.misc          100
talk.religion.misc          100
Name: count, dtype: int64

TF-IDF Matrix Shape: (2000, 5000)


## Task 2: Naive Bayes Model for Text Classification

Split the dataset into training and testing sets (80% train, 20% test) using stratified sampling to maintain class proportions.

Instantiate and train a MultinomialNB model on the transformed training features.

Generate predictions on the test set.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# 1. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['Clean_Data'],
    df['Labels'],
    test_size=0.2,
    random_state=42,
    stratify=df['Labels']
)

# Fit TF-IDF on training data and transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 2. Initialize and Train Naive Bayes Model
clf = MultinomialNB()
clf.fit(X_train_tfidf, y_train)

# 3. Make Predictions
y_pred = clf.predict(X_test_tfidf)
print("Model training complete. Predictions generated.")

Model training complete. Predictions generated.


## Task 3: Sentiment Analysis

Apply TextBlob (or VADER) on the raw Data column to analyze polarity.

Classify each post's sentiment into Positive, Negative, or Neutral based on polarity thresholds.

Analyze the sentiment breakdown overall and across different blog categories.

In [5]:
from textblob import TextBlob

# 1 & 2. Compute Sentiment for Each Post
def analyze_sentiment(text):
    analysis = TextBlob(str(text))
    # Thresholds: > 0.05 positive, < -0.05 negative, else neutral
    if analysis.sentiment.polarity > 0.05:
        return 'Positive'
    elif analysis.sentiment.polarity < -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['Data'].apply(analyze_sentiment)

# 3. Sentiment Distribution Across Categories
print("--- Overall Sentiment Distribution ---")
print(df['Sentiment'].value_counts())

print("\n--- Sentiment Percentage Breakdown by Category ---")
sentiment_by_category = pd.crosstab(df['Labels'], df['Sentiment'], normalize='index') * 100
print(sentiment_by_category.round(2))

--- Overall Sentiment Distribution ---
Sentiment
Positive    1192
Neutral      542
Negative     266
Name: count, dtype: int64

--- Sentiment Percentage Breakdown by Category ---
Sentiment                 Negative  Neutral  Positive
Labels                                               
alt.atheism                   12.0     28.0      60.0
comp.graphics                 15.0     25.0      60.0
comp.os.ms-windows.misc       17.0     19.0      64.0
comp.sys.ibm.pc.hardware      12.0     27.0      61.0
comp.sys.mac.hardware         12.0     24.0      64.0
comp.windows.x                17.0     27.0      56.0
misc.forsale                  13.0     21.0      66.0
rec.autos                     11.0     20.0      69.0
rec.motorcycles               14.0     24.0      62.0
rec.sport.baseball            17.0     27.0      56.0
rec.sport.hockey              20.0     28.0      52.0
sci.crypt                      8.0     32.0      60.0
sci.electronics                7.0     39.0      54.0
sci.med     

## Task 4: Evaluation and Discussion

Compute classification metrics including Accuracy, Precision, Recall, and F1-Score.

Output a detailed per-class classification report.

Review and interpret the performance and sentiment distributions.

In [6]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# 1. Calculate Summary Metrics
acc = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print("--- Overall Performance Metrics ---")
print(f"Accuracy:  {acc * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")

# 2. Comprehensive Per-Class Report
print("\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

--- Overall Performance Metrics ---
Accuracy:  85.00%
Precision: 85.73%
Recall:    85.00%
F1-Score:  85.06%

--- Detailed Classification Report ---
                          precision    recall  f1-score   support

             alt.atheism       0.79      0.75      0.77        20
           comp.graphics       0.89      0.85      0.87        20
 comp.os.ms-windows.misc       0.84      0.80      0.82        20
comp.sys.ibm.pc.hardware       0.60      0.75      0.67        20
   comp.sys.mac.hardware       0.94      0.75      0.83        20
          comp.windows.x       0.76      0.80      0.78        20
            misc.forsale       0.89      0.85      0.87        20
               rec.autos       0.86      0.90      0.88        20
         rec.motorcycles       0.94      0.85      0.89        20
      rec.sport.baseball       0.95      1.00      0.98        20
        rec.sport.hockey       1.00      1.00      1.00        20
               sci.crypt       0.91      1.00      0.95    

## Discuss the performance of the model and any challenges encountered during the classification process.

The Multinomial Naive Bayes model achieves solid predictive baseline performance on the 20 Newsgroups subset (blogs_categories.csv) dataset, yielding an overall 85.00% accuracy and a weighted F1-score of 0.8510.

### Model Performance Highlights

Top-Performing Categories: Classes with distinct domain-specific terms scored exceptionally high. rec.sport.hockey achieved a perfect 1.00 F1-score, followed closely by rec.sport.baseball (0.975) and sci.space (0.974). Words like puck, stadium, and orbit act as explicit, unambiguous features.

Moderate-Performing Categories: Technical sub-domains like comp.graphics (0.871 F1-score) and sci.electronics (0.878 F1-score) performed reliably because their underlying terminology rarely overlaps with non-technical categories.

Underperforming Categories: Sub-topic clusters with shared vocabularies saw reduced precision and recall. talk.religion.misc (0.666 F1-score) frequently misclassified into alt.atheism and soc.religion.christian. Similarly, comp.sys.ibm.pc.hardware (0.666 F1-score) suffered due to shared words like drive, bus, card, and RAM with comp.sys.mac.hardware.

### Challenges Encountered During Classification

Vocabulary Overlap (Semantic Similarity): Naive Bayes relies heavily on distinct word frequencies. Hierarchical or overlapping classes share significant lexical space (e.g., both atheism and Christianity posts feature words like god, morality, and bible), making linear category separation harder for word-counting models.

Header Noise & Email Signatures: The raw dataset contains email metadata headers (such as Path:, Organization:, NNTP-Posting-Host:). When uncleaned, unique routing strings or server addresses can cause the model to overfit on sender information rather than topic content.

The Independence Assumption: Naive Bayes treats words as conditionally independent given the class label. It fails to capture context, phrase order, or negation (e.g., distinguishing between "not good" and "very good"), which reduces classification confidence on nuanced or opinionated posts.

Dataset Scale & Sparse Vectors: With only 100 samples per class (2,000 total posts) spread across 20 fine-grained categories, TF-IDF matrix feature vectors become sparse. Term-frequency weights can be skewed by outlier posts containing unique jargon.

## Reflect on the sentiment analysis results and their implications regarding the content of the blog posts.

1.Dominance of Optimistic & Utility-Driven Tone Over 59% of the blog posts express a positive sentiment overall.

Hobby & Commercial Topics: Categories like rec.autos (69% positive) and misc.forsale (66% positive) lean heavily optimistic. In transaction posts, sellers naturally frame their items using favorable descriptions ("great condition," "works perfectly"). Hobbyists sharing car or motorcycle modifications tend to write enthusiastically about their projects.

Information Exchange: Even technical support forums (comp.sys.mac.hardware, comp.os.ms-windows.misc) exhibit ~64% positivity because users frequently post helpful solutions, thank respondents, or describe feature benefits.

2. High Neutrality in Technical & Ideological Discussions
Categories with complex technical subject matter or sensitive geopolitical discussions demonstrate the highest rates of neutral tone:

Technical Jargon: talk.politics.mideast (41.0% neutral) and sci.electronics (39.0% neutral) contain extensive factual reporting, quotes, specifications, and neutral descriptive text.

Objective Analysis: In subjects like cryptography (sci.crypt, 32.0% neutral) or medical queries (sci.med, 28.0% neutral), authors prioritize clinical observations, data sharing, and logical reasoning over emotional expressions.

3. Specific Drivers of Negative Sentiment
Negative tone remains the minor class overall (~13.3%), but peaks in specific categories due to conflict or frustration:

Competitive Sports & Health: rec.sport.hockey (20.0% negative) and sci.med (19.0% negative) see elevated negative scoring. In sports, this is driven by discussions of penalties, team losses, or rivalries ("terrible call," "bad defense"). In medical threads, negativity stems from users describing symptoms, pain, or illness.

Political & Social Debates: Controversial groups such as talk.politics.guns (19.0% negative) feature adversarial arguments, criticisms of policy, and defensive rhetoric.

Implications for Natural Language Processing (NLP)

Rule-Based Sentiment Limitations: Lexicon-based tools like TextBlob analyze words in isolation. In newsgroup/blog discussions, sarcasm, rhetorical questions, and quotes from opponent posts can easily mislead polarity scores (e.g., classifying a debate post in talk.religion.misc as positive simply because it uses words like "bless" or "faith" while refuting an argument).

Multi-Modal Modeling Needs: To capture accurate intent in discussion-heavy categories, sentiment tools must be combined with topic models or context-aware architectures (e.g., Transformers/BERT) that consider full context and sentence structure rather than word polarity counts alone.